# 时序异常检测工程：季节基线、阈值、事件指标与流式状态

时序异常检测不是对整列数据调用一次 z-score。本 Notebook 从监控场景实现：时间/时区与重复点合同、缺失处理、严格时间切分、季节基线、MAD 鲁棒分数、validation 阈值、point 与 range/event 指标、告警合并、漂移、流式去重/乱序和版本化发布。

## 学习目标

1. 处理 timestamp、采样频率、迟到/重复/缺失和标签区间；
2. 只用过去拟合季节基线与尺度，避免未来泄漏；
3. 用 validation 选择阈值，test 只做冻结评估；
4. 区分点指标、事件命中、告警数量和检测延迟；
5. 设计有状态、幂等、可降级的在线 detector。

> 合成异常幅度较大，结果会很漂亮，只说明这套受控路径可执行；真实异常通常弱、漂移、多形态且标签不完整。

## 1. 数据与告警合同

输入至少包含 `series_id、tenant、event_time(带时区)、value、quality、event_id、ingested_at`。要声明采样频率、聚合语义（sum/mean/last）、允许迟到、重复覆盖规则、缺失含义和单位。标签应是 `[start,end)` 区间并记录来源；“没有标签”不等于正常。

输出不是裸分数，而是 `score、threshold、is_anomaly、event_id、baseline/model/version、reason、degraded`。告警层还要合并相邻点、抑制风暴、路由 owner，并保留确认/误报反馈。

In [ ]:
from __future__ import annotations

from dataclasses import dataclass, field
from datetime import datetime, timedelta, timezone
import hashlib
import json
import math
import numpy as np
import pandas as pd

RNG = np.random.default_rng(31)
N = 24 * 28
start = pd.Timestamp("2026-01-01T00:00:00Z")
t = np.arange(N)
hour = t % 24
value = 100 + 12 * np.sin(2 * np.pi * hour / 24) + RNG.normal(0, 1.0, N)
error_rate = 0.02 + 0.004 * np.sin(2 * np.pi * (hour + 4) / 24) + RNG.normal(0, 0.001, N)
label = np.zeros(N, dtype=int)
true_ranges = [(389, 395, 9), (474, 478, -8), (555, 562, 10), (638, 644, -9)]
for left, right, magnitude in true_ranges:
    value[left:right] += magnitude
    error_rate[left:right] += 0.012
    label[left:right] = 1

base = pd.DataFrame({
    "series_id": "api-latency", "tenant": "tenant-a",
    "event_time": pd.date_range(start, periods=N, freq="h"),
    "value": value, "error_rate": error_rate, "label": label,
    "event_id": [f"metric-{i:04d}" for i in range(N)],
})
base["ingested_at"] = base.event_time + pd.Timedelta(minutes=2)
for index in (50, 200, 520):
    base.loc[index, ["value", "error_rate"]] = np.nan
duplicate = base.iloc[[100]].copy()
duplicate["event_id"] = "metric-0100-correction"
duplicate["value"] += 0.2
duplicate["ingested_at"] += pd.Timedelta(minutes=3)
raw = pd.concat([base, duplicate], ignore_index=True).sample(frac=1, random_state=7).reset_index(drop=True)
print("raw rows=", len(raw), "unique timestamps=", raw.event_time.nunique())


## 2. 摄取质量：重复、缺失与时区先于模型

同一 timestamp 多条记录时，本例按 `ingested_at` 取最新 correction；生产要用 source sequence 或 revision，不能只凭到达时间。重建完整 hourly grid 后，短缺口最多前向填充 2 点，并保留 `was_missing`，长缺口应降级/拒绝而非凭空插值。

前向填充是因果的；双向插值会用未来值。求和型计数与平均型 gauge 的补值策略不同。DST、本地时区和 leap second 也必须在边界层统一处理。

In [ ]:
clean = (raw.sort_values(["event_time", "ingested_at"])
         .drop_duplicates("event_time", keep="last")
         .set_index("event_time")
         .asfreq("h"))
clean["series_id"] = clean.series_id.ffill().bfill()
clean["tenant"] = clean.tenant.ffill().bfill()
clean["label"] = clean.label.fillna(0).astype(int)
clean["was_missing"] = clean[["value", "error_rate"]].isna().any(axis=1)
clean[["value", "error_rate"]] = clean[["value", "error_rate"]].ffill(limit=2)
if clean[["value", "error_rate"]].isna().any().any():
    raise ValueError("存在超过允许长度的缺口")
clean = clean.reset_index()
assert len(clean) == N and clean.event_time.is_monotonic_increasing
assert int(clean.was_missing.sum()) == 3
assert clean.event_time.dt.tz is not None
print(clean[["event_time", "value", "error_rate", "was_missing"]].head())


## 3. 时间切分：train → validation → test

前 14 天 train 拟合基线和尺度，接着 7 天 validation 选 threshold，最后 7 天 test 只评一次。随机切分会把未来季节形态、漂移甚至同一异常区间拆进训练。窗口特征也必须只用当前及过去点。

本例 train 没有注入异常，实际训练集可能被污染，应使用鲁棒统计、迭代清洗和审计，而不是假设完全干净。

In [ ]:
index = np.arange(len(clean))
train_mask = index < 24 * 14
valid_mask = (index >= 24 * 14) & (index < 24 * 21)
test_mask = index >= 24 * 21
assert clean.loc[train_mask, "label"].sum() == 0
assert clean.loc[valid_mask, "label"].sum() == 10
assert clean.loc[test_mask, "label"].sum() == 13
assert clean.loc[train_mask, "event_time"].max() < clean.loc[valid_mask, "event_time"].min()
assert clean.loc[valid_mask, "event_time"].max() < clean.loc[test_mask, "event_time"].min()
print({"train": int(train_mask.sum()), "validation": int(valid_mask.sum()), "test": int(test_mask.sum())})


## 4. 季节基线与 MAD 鲁棒尺度

对 hourly seasonality，本例仅用 train 计算每个小时的中位数；残差再用全局 median 与 $1.4826\times MAD$ 标准化。MAD 比均值/标准差不易被少量尖峰拖动。真实系统可用 STL、状态空间、预测模型或多季节模型，但必须保证在线预测只使用 cutoff 前的数据。

多变量分数取 latency 与 error rate 的最大鲁棒 z；这是可解释 OR 规则，不代表联合概率。

In [ ]:
clean["hour"] = clean.event_time.dt.hour

def fit_robust_seasonal(values: np.ndarray):
    seasonal = np.array([np.median(values[train_mask & (clean.hour.to_numpy() == h)]) for h in range(24)])
    residual = values - seasonal[clean.hour.to_numpy()]
    center = float(np.median(residual[train_mask]))
    mad = float(np.median(np.abs(residual[train_mask] - center)))
    scale = max(1.4826 * mad, 1e-9)
    return seasonal, center, scale, residual

value_season, value_center, value_scale, value_residual = fit_robust_seasonal(clean.value.to_numpy())
error_season, error_center, error_scale, error_residual = fit_robust_seasonal(clean.error_rate.to_numpy())
value_score = np.abs(value_residual - value_center) / value_scale
error_score = np.abs(error_residual - error_center) / error_scale
anomaly_score = np.maximum(value_score, error_score)
assert np.isfinite(anomaly_score).all()
print({"value_scale": value_scale, "error_scale": error_scale})


## 5. 阈值只能在 validation 选择

阈值体现误报成本与漏报成本，不是固定的 3-sigma 定理。这里遍历候选 threshold，以 validation point-F1 最大为准，并在并列时选更低阈值保护 recall；随后冻结到 test。生产还要按 series/时段分桶、给阈值置信区间，并限制一次调参反复看 test。

In [ ]:
def binary_metrics(y_true: np.ndarray, predicted: np.ndarray):
    tp = int(np.sum((y_true == 1) & predicted))
    fp = int(np.sum((y_true == 0) & predicted))
    fn = int(np.sum((y_true == 1) & ~predicted))
    precision = tp / (tp + fp) if tp + fp else 0.0
    recall = tp / (tp + fn) if tp + fn else 0.0
    f1 = 2 * precision * recall / (precision + recall) if precision + recall else 0.0
    return {"precision": precision, "recall": recall, "f1": f1, "tp": tp, "fp": fp, "fn": fn}

threshold_rows = []
y_valid = clean.loc[valid_mask, "label"].to_numpy()
for threshold in np.arange(2.5, 6.6, 0.5):
    result = binary_metrics(y_valid, anomaly_score[valid_mask] >= threshold)
    threshold_rows.append({"threshold": float(threshold), **result})
selected = max(threshold_rows, key=lambda row: (row["f1"], -row["threshold"]))
THRESHOLD = selected["threshold"]
print(pd.DataFrame(threshold_rows).round(3))
print("frozen threshold=", THRESHOLD)
assert THRESHOLD >= 2.5


## 6. Point 指标不等于告警体验

point precision/recall 会让长异常贡献很多 TP，也会把一次持续告警算成几十次。range/event 指标要回答：每个真实事件是否被至少一个告警命中、每个告警是否对应真实事件、首次检测延迟多大。容忍窗口、存在性奖励和持续长度权重必须按业务写清。

In [ ]:
frozen_prediction = anomaly_score >= THRESHOLD
test_result = binary_metrics(clean.loc[test_mask, "label"].to_numpy(), frozen_prediction[test_mask])
print("test point metrics:", test_result)
assert test_result["recall"] >= 0.9
assert test_result["precision"] >= 0.8


## 7. 把连续点合并成事件

下面把连续 True 点转成 `[start,end)` 索引区间，再以任意重叠作为命中。真实告警常允许前后 tolerance、要求最小持续点数并设置 cooldown；这些后处理规则必须版本化，因为它们会显著改变告警数和指标。

In [ ]:
def contiguous_ranges(mask: np.ndarray, offset=0) -> list[tuple[int, int]]:
    ranges, start_index = [], None
    for local, active in enumerate(mask):
        if active and start_index is None:
            start_index = local + offset
        if not active and start_index is not None:
            ranges.append((start_index, local + offset))
            start_index = None
    if start_index is not None:
        ranges.append((start_index, len(mask) + offset))
    return ranges

def overlap(left: tuple[int, int], right: tuple[int, int], tolerance=0) -> bool:
    return left[0] < right[1] + tolerance and right[0] < left[1] + tolerance

def event_metrics(true_events, predicted_events, tolerance=1):
    recall = np.mean([any(overlap(event, alert, tolerance) for alert in predicted_events) for event in true_events]) if true_events else 0.0
    precision = np.mean([any(overlap(alert, event, tolerance) for event in true_events) for alert in predicted_events]) if predicted_events else 0.0
    return {"event_precision": float(precision), "event_recall": float(recall), "alerts": len(predicted_events)}

test_offset = int(np.flatnonzero(test_mask)[0])
true_test_events = contiguous_ranges(clean.loc[test_mask, "label"].to_numpy().astype(bool), test_offset)
predicted_test_events = contiguous_ranges(frozen_prediction[test_mask], test_offset)
event_result = event_metrics(true_test_events, predicted_test_events)
print({"true_events": true_test_events, "predicted_events": predicted_test_events, **event_result})
assert event_result["event_recall"] == 1.0


## 8. 漂移与模型失效

输入均值变化、季节形态变化、采样频率改变、单位切换都会让旧阈值失效。监控缺失率、残差中位数/MAD、分数分布、告警率和有延迟真值后的 precision/recall。漂移告警只说明分布变了，不自动证明模型坏了；需要回放、抽样标注和根因分析。

In [ ]:
def population_stability_index(reference: np.ndarray, current: np.ndarray, bins=8, epsilon=1e-6):
    edges = np.unique(np.quantile(reference, np.linspace(0, 1, bins + 1)))
    edges[0], edges[-1] = -np.inf, np.inf
    ref = np.histogram(reference, bins=edges)[0] / len(reference)
    cur = np.histogram(current, bins=edges)[0] / len(current)
    ref, cur = np.clip(ref, epsilon, None), np.clip(cur, epsilon, None)
    return float(np.sum((cur - ref) * np.log(cur / ref)))

reference_score = anomaly_score[train_mask]
shifted_current = reference_score + 3.0
score_psi = population_stability_index(reference_score, shifted_current)
print({"score_PSI_demo": score_psi, "missing_rate": float(clean.was_missing.mean())})
assert score_psi > 0


## 9. 流式 detector：状态、幂等与乱序

在线 detector 需要 series-scoped 状态：watermark、已处理 event_id、payload 指纹、最后值、seasonal baseline、模型版本。所有带时区时间先归一到 UTC，再参与季节槽位、指纹与 watermark 比较；同一瞬间的不同 offset 表示应视为相同。相同 event_id 与相同 payload 返回幂等结果；相同 ID 却不同 payload 必须拒绝并走显式 correction 流程。超过允许迟到窗口的点进入补算/死信，不直接改写已发告警。状态升级要有 schema 版本和回滚。

下面只演示单 tenant、单 series 的安全边界。

In [ ]:
@dataclass
class StreamDetector:
    tenant: str
    threshold: float
    allowed_lateness: timedelta = timedelta(hours=2)
    watermark: datetime | None = None
    seen: dict[str, tuple[str, dict]] = field(default_factory=dict)

    @staticmethod
    def payload_fingerprint(event: dict) -> str:
        timestamp_utc = event["event_time"].astimezone(timezone.utc)
        canonical = "|".join([
            event["tenant"], timestamp_utc.isoformat(),
            format(float(event["value"]), ".17g"), format(float(event["error_rate"]), ".17g"),
        ])
        return hashlib.sha256(canonical.encode()).hexdigest()

    def process(self, event: dict) -> dict:
        if event["tenant"] != self.tenant:
            raise PermissionError("tenant 不匹配")
        timestamp = event["event_time"]
        if timestamp.tzinfo is None or timestamp.utcoffset() is None:
            raise ValueError("event_time 必须带有效时区偏移")
        timestamp = timestamp.astimezone(timezone.utc)
        fingerprint = self.payload_fingerprint(event)
        if event["event_id"] in self.seen:
            stored_fingerprint, stored_result = self.seen[event["event_id"]]
            if stored_fingerprint != fingerprint:
                raise ValueError("同一 event_id 出现不同 payload，必须走显式 correction 流程")
            return {**stored_result, "duplicate": True}
        if self.watermark and timestamp < self.watermark - self.allowed_lateness:
            raise ValueError("事件超过允许迟到窗口")
        h = timestamp.hour
        value_z = abs((event["value"] - value_season[h]) - value_center) / value_scale
        error_z = abs((event["error_rate"] - error_season[h]) - error_center) / error_scale
        score = float(max(value_z, error_z))
        result = {"score": score, "is_anomaly": score >= self.threshold, "duplicate": False, "model_version": "seasonal-mad-v1"}
        self.seen[event["event_id"]] = (fingerprint, result)
        self.watermark = max(self.watermark, timestamp) if self.watermark else timestamp
        return result

detector = StreamDetector("tenant-a", THRESHOLD)
live_event = {"event_id": "live-1", "tenant": "tenant-a", "event_time": datetime(2026, 2, 1, 12, tzinfo=timezone.utc), "value": 130.0, "error_rate": 0.05}
first_live = detector.process(live_event)
second_live = detector.process(live_event)
equivalent_offset_event = {**live_event, "event_time": datetime(2026, 2, 1, 20, tzinfo=timezone(timedelta(hours=8)))}
third_live = detector.process(equivalent_offset_event)
assert first_live["is_anomaly"] and second_live["duplicate"] and third_live["duplicate"]


## 10. 告警运营、降级与成本

检测点先经最小持续、merge gap、cooldown 和维护窗口，再通知；每个规则都进入 trace。下游通知失败不能丢事件，要有 durable queue 与重试幂等键。基线服务不可用时可退回静态阈值，但必须标记 degraded；数据质量失效时应 fail closed 或发“数据缺失”告警，不能把 NaN 当正常。

容量估算包括 series 数、每秒点数、状态大小、回放窗口、特征计算和告警峰值。高基数标签会同时压垮时序库和 detector。

In [ ]:
baseline_payload = {
    "model_version": "seasonal-mad-v1",
    "frequency": "1h",
    "train_end": clean.loc[train_mask, "event_time"].max().isoformat(),
    "threshold": THRESHOLD,
    "threshold_selection": "validation-point-f1-lowest-tie",
    "aggregation": "gauge-last-with-revision",
    "missing_policy": "causal-ffill-limit-2",
    "event_postprocess": {"merge_gap": 0, "tolerance": 1},
    "value_seasonal": value_season.round(8).tolist(),
    "error_seasonal": error_season.round(8).tolist(),
}
manifest_blob = json.dumps(baseline_payload, sort_keys=True).encode()
baseline_payload["bundle_sha256"] = hashlib.sha256(manifest_blob).hexdigest()
assert len(baseline_payload["bundle_sha256"]) == 64
print(json.dumps({k: v for k, v in baseline_payload.items() if "seasonal" not in k}, ensure_ascii=False, indent=2))


## 11. 回归门禁

必须覆盖时区、重复 revision、缺失上限、时间切分、train-only baseline、MAD 零尺度、阈值冻结、区间边界、告警合并、重复消息、乱序、tenant 和 manifest。另需用“正常但发生节假日/发布/单位变化”的数据做误报回归，避免只测明显尖峰。

In [ ]:
assert raw.event_time.duplicated().sum() == 1
assert not clean.event_time.duplicated().any()
assert value_scale > 0 and error_scale > 0
assert len(value_season) == len(error_season) == 24
assert selected["f1"] == max(row["f1"] for row in threshold_rows)
assert len(true_test_events) == 2
assert all(left < right for left, right in predicted_test_events)
assert detector.watermark == live_event["event_time"]
try:
    detector.process({**live_event, "event_id": "wrong-tenant", "tenant": "tenant-b"})
    raise AssertionError("跨 tenant 必须拒绝")
except PermissionError:
    pass
try:
    detector.process({**live_event, "value": 131.0})
    raise AssertionError("同一 event_id 的冲突 payload 必须拒绝")
except ValueError:
    pass
try:
    detector.process({**live_event, "event_id": "naive-time", "event_time": datetime(2026, 2, 1, 12)})
    raise AssertionError("无时区事件必须拒绝")
except ValueError:
    pass
try:
    detector.process({**live_event, "event_id": "too-late", "event_time": datetime(2026, 1, 1, tzinfo=timezone.utc)})
    raise AssertionError("超迟事件必须拒绝")
except ValueError:
    pass
assert baseline_payload["train_end"] < clean.loc[test_mask, "event_time"].min().isoformat()
print("时序摄取、切分、基线、阈值、事件指标、流式状态和 tenant 断言全部通过。")


## 12. 研究依据与教学边界

- Cleveland et al., *STL: A Seasonal-Trend Decomposition Procedure Based on Loess*, Journal of Official Statistics, 1990。
- Liu, Ting & Zhou, [Isolation Forest](https://doi.org/10.1109/ICDM.2008.17), ICDM 2008。
- Tatbul et al., [Precision and Recall for Time Series](https://proceedings.neurips.cc/paper/2018/hash/8f468c873a32bb0619eaeb2050ba45d1-Abstract.html), NeurIPS 2018：区间异常评估。
- scikit-learn 官方文档：[Outlier detection](https://scikit-learn.org/stable/modules/outlier_detection.html)，可用于比较成熟 baseline。

本例没有 STL 拟合、Isolation Forest、预测区间、多 series 分层、迟到补算存储或通知系统；季节 median+MAD 只是一条透明基线。生产替换算法后，仍要保留时间合同、validation 阈值、事件指标、状态幂等与降级测试。